In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "catalog_project")
dbutils.widgets.text("schemaOrigin", "bronze")
dbutils.widgets.text("schemaDest", "silver")


In [0]:

catalog = dbutils.widgets.get("catalog")
schemaOrigin = dbutils.widgets.get("schemaOrigin")
schemaDest = dbutils.widgets.get("schemaDest")

In [0]:

df_accounts = spark.table(f"{catalog}.{schemaOrigin}.accounts")
df_fileppe = spark.table(f"{catalog}.{schemaOrigin}.fileppe")
df_product = spark.table(f"{catalog}.{schemaOrigin}.product")
df_client = spark.table(f"{catalog}.{schemaOrigin}.client")

In [0]:
df_accounts_sel= df_accounts.select("idProducto",
                                          "idCliente",
                                          "idNoCta")                                      
                                                                                

In [0]:
df_file_ppe = df_fileppe.select("cardNumber",
                                    "account",
                                    "amount"
                                    )

In [0]:
df_product_select = df_product.select("idProducto",
                                          "descripcion")           

In [0]:
df_client_select = df_client.select("idProducto",
                                    "idCliente",
                                    "Nombre",
                                    "excenta",
                                    "ingestion_date"
                                    )

In [0]:

df_accountsbyfile = df_accounts_sel.alias("ac").join(df_file_ppe .alias("fp"), col("ac.idNoCta") == col("fp.account"), "inner")
df_productbyclient = df_product_select.alias("p").join(df_client_select .alias("c"), col("p.idproducto") == col("c.idProducto"), "inner")
                                                         

In [0]:
df_productbyclientselect = df_productbyclient.select(col("p.idproducto"),
                                                            col("p.descripcion"), 
                                                            col("c.idCliente"),
                                                            col("c.Nombre"), 
                                                            col("c.excenta"),
                                                            col("c.ingestion_date")).orderBy("p.idproducto")

In [0]:
df_productbyclientselect.display()

In [0]:
df_accountfileppe = df_accountsbyfile.select(col("ac.idproducto"),
                                             col("ac.idCliente"),
                                             col("ac.idNoCta"), 
                                            col("fp.cardNumber"), 
                                            col("fp.amount")).orderBy("ac.idProducto")  

In [0]:
df_accountfileppe.display()

In [0]:

df_accfileproduct = df_accountsbyfile.alias("af").join(df_productbyclientselect.alias("pc"), (col("af.idproducto") == col("pc.idproducto")) & (col("af.idCliente") == col("pc.idCliente")), "inner")                               
                                                            

In [0]:
df_accfileproduct.display()

In [0]:
df_agrupar = df_accfileproduct.groupBy("af.idproducto","excenta").agg(sum("amount").alias("total")).orderBy("af.idproducto")

In [0]:
df_agrupar.display()

In [0]:
df_agrupar.write.mode("overwrite").insertInto(f"{catalog}.{schemaDest}.totalbyproduct")